# Assignment 5: Neural Networks

---

## Task 1) RNN as Language Model

Similar to the n-gram language models in the previous tasks, imagine you'd have to write another thesis and just want to generate an interesting topic.
In this assignment, you will train and use Recurrent Neural Networks as language models to generate new potential thesis topics.

### Data

Download the `theses.csv` data set from the `Supplemental Materials` in the `Files` section of our Microsoft Teams group.
This dataset consists of approx. 3,000 theses topics chosen by students in the past.
Here are some examples of the file content:

```
27.10.94;14.07.95;1995;intern;Diplom;DE;Monte Carlo-Simulation für ein gekoppeltes Round-Robin-System;
04.11.94;14.03.95;1995;intern;Diplom;DE;Implementierung eines Testüberdeckungsgrad-Analysators für RAS;
01.11.20;01.04.21;2021;intern;Bachelor;DE;Landessprachenerkennung mittels X-Vektoren und Meta-Klassifikation;
```

### Basic Setup

For the assignment on Recurrent Neural Networks, we'll (again) heavily use [PyTorch](https://pytorch.org) as go-to Deep Learning library.
Here, we'll rely on the RNN and Embedding modules already implemented by PyTorch.
You can imagine the Embedding layer as a simple lookup table that stores embeddings of a fixed dictionary and size (quite similar to the Word2Vec parameters we've trained in assignment 2).
Head over to the [RNN](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html) and [Embedding](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html) modules to gain some understanding of their functionality.
Code for processing data samples, batching, converting to tensors, etc. can get messy and hard to maintain. 
Therefore, you can use PyTorch's [Datasets & DataLoaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html). 
Get familiar with the basics of data handling, as it will help you for upcoming assignments.
As always, you can use [NumPy](https://numpy.org) and [Pandas](https://pandas.pydata.org) for data handling etc.

*In this Jupyter Notebook, we will provide the steps to solve this task and give hints via functions & comments. However, code modifications (e.g., function naming, arguments) and implementation of additional helper functions & classes are allowed. The code aims to help you get started.*

---

In [1]:
# Dependencies
import os
import tqdm
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

### Prepare the Data

1.1 Spend some time on preparing the dataset. It may be helpful to lower-case the data and to filter for German titles. The format of the CSV-file should be:

```
Anmeldedatum;Abgabedatum;JahrAkademisch;Art;Grad;Sprache;Titel;Abstract
```

1.2 Create the vocabulary from the prepared dataset. You'll need it for the modeling part such as nn.Embedding.

1.3 Create a PyTorch Dataset class which handles your tokenized data with respect to model inputs and labels.

In [2]:
def load_theses_dataset(filepath):
    """Loads all theses instances and returns them as a dataframe."""
    ### YOUR CODE HERE
    
    return pd.read_csv(filepath, header=0, sep=";")
    
    ### END YOUR CODE

In [3]:
def preprocess(dataframe, start_token="<s>", end_token="</s>"):
    """Preprocesses and tokenizes the given theses titles for further use."""
    ### YOUR CODE HERE
    
    dataframe = dataframe.copy()
    
    # lowercase titles
    dataframe["Titel"] = dataframe["Titel"].str.lower()

    # add sentence start and end symbol to titles
    dataframe["Titel"] = dataframe["Titel"].apply(lambda x: f"{start_token} {x} {end_token}")

    # simple tokenization of titles
    dataframe["tokenized"] = [title.split() for title in dataframe["Titel"].values]

    return dataframe

    ### END YOUR CODE

In [4]:
# dataframe = load_theses_dataset(...)
# tokenized_data = preprocess(dataframe)
# vocabulary = ...
# word2idx = ...
# idx2word = ...

dataframe = load_theses_dataset("data/theses2022.csv")

dataframe = dataframe[dataframe["Sprache"] == "DE"]

dataframe = preprocess(dataframe)

print(f"Num theses: {len(dataframe)}")

Num theses: 2982


In [5]:
vocab = set()
for s in dataframe.tokenized:
    vocab.update(s)

vocab_size = len(vocab)

word2idx = {w: idx for (idx, w) in enumerate(sorted(vocab))}
idx2word = {idx: w for (idx, w) in enumerate(sorted(vocab))}

print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 9339


In [6]:
class ThesisDataset(Dataset):
    def __init__(self, dataset, word2idx):
        self.data, self.labels = [], []
        for tokens in dataset:
            # Create inputs
            self.data.append(torch.stack([
                torch.tensor(word2idx[w],dtype=torch.long) for w in tokens
            ]))

            # Create labels
            self.labels.append(torch.stack([
                torch.tensor(word2idx[w], dtype=torch.long) for w in tokens
            ]))


    def __len__(self):
        return len(self.data)


    def __getitem__(self, idx):
        # Remove </s> token as this is the last possible prediction
        sample = self.data[idx][:-1]

        # Ensure next token prediction and remove last token (first token due to `roll`)
        labels = self.labels[idx].roll(-1)[:-1]
        return sample, labels

### Train and Evaluate

2.1 Implement the RNN Language Model. Therefore, you can use the nn.Module and overwrite the forward function. For the embedding layer you can either use the embeddings learned from the previous word2vec assignment or train the `nn.Embedding` module and corresponding parameters from scratch.

2.2 Implement the functionality to train your model with the train dataset.

2.3 Implement the functionality to evaluate your model with the test dataset.

2.4 Perform a train-test-split for your theses data, train the RNN Language Model and evaluate the loss & perplexity.

In [7]:
### TODO: 2.1 Implement RNN Language Model (nn.Module)

### YOUR CODE HERE

class RNN_LM(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, hidden_dim, num_rnn_layers=1):
        super(RNN_LM, self).__init__()

        self.embedding = nn.Embedding(
            num_embeddings=num_embeddings,
            embedding_dim=embedding_dim
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_rnn_layers,
            dropout=0.2
        )

        self.fc = nn.Linear(hidden_dim, num_embeddings)

    
    def forward(self, X, hidden=None):
        embeddings = self.embedding(X)

        outputs, hidden = self.rnn(embeddings, hidden)

        logits = self.fc(outputs)

        return logits, hidden

### END YOUR CODE

In [8]:
### TODO: 2.2 Implement the train functionality
### Notice: If you want, you can also combine train and eval

### YOUR CODE HERE

def train(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0

    optimizer.zero_grad()

    for inputs, labels in tqdm.tqdm(dataloader, desc="Train"):
        inputs = inputs.to(device)
        labels = labels.to(device)

        logits, hidden = model(inputs)

        loss = criterion(logits.view(*logits.shape[1:]), labels.view(*labels.shape[1:]))

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        running_loss += loss.item() * inputs.size(0)

    running_loss = running_loss / len(dataloader)
    return running_loss

### END YOUR CODE

In [9]:
### TODO: 2.3 Implement the evaluation functionality
### Notice: If you want, you can also combine train and eval

### YOUR CODE HERE

def eval(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            logits, hidden = model(inputs)

            loss = criterion(logits.view(*logits.shape[1:]), labels.view(*labels.shape[1:]))

            running_loss += loss.item()

    running_loss = running_loss / len(dataloader)
    return running_loss, torch.exp(torch.tensor(running_loss))

### END YOUR CODE

In [10]:
### TODO: 2.4 Initialize and train the RNN Language Model for X epochs

# For split reproducibility
SEED = 42

EPOCHS = 100

DEVICE = "cpu" # 'cpu', 'mps' or 'cuda'

train_data, test_data = train_test_split(
    dataframe.tokenized, 
    test_size=int(0.2 * len(dataframe)),
    shuffle=True, random_state=SEED
)

### YOUR CODE HERE

# Use batch_size=1 to avoid padding
train_dataset = ThesisDataset(train_data, word2idx=word2idx)
train_dataloader = DataLoader(train_dataset, batch_size=1)

# Use batch_size=1 to avoid padding
test_dataset = ThesisDataset(test_data, word2idx=word2idx)
test_dataloader = DataLoader(test_dataset, batch_size=1)


model = RNN_LM(
    num_embeddings=len(vocab),
    embedding_dim=32,
    hidden_dim=32,
    num_rnn_layers=1
)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(reduction="mean")

optimizer = optim.SGD(model.parameters(), lr=0.01)


best_epoch = -1
best_loss = np.Inf
for epoch in range(1, EPOCHS + 1):
    print(f"Epoch {epoch} of {EPOCHS}")
    print("-" * 20)

    # Training step
    train_loss = train(
        model=model,
        dataloader=train_dataloader,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE
    )

    # Evaluation step
    test_loss, ppl = eval(
        model=model,
        dataloader=test_dataloader,
        criterion=criterion,
        device=DEVICE
    )

    print(f"Train loss: {train_loss:.4f} | Test loss: {test_loss:.4f} | Perplexity: {ppl:.4f}")

    # Save best model by evaluation loss
    if test_loss <= best_loss:
        best_epoch = epoch
        best_loss = test_loss
        torch.save(model.state_dict(), "data/best_rnn_lm.pt")


print(f"Best epoch: {best_epoch}")
print(f"Best test loss: {best_loss:.4f}")
print(f"Loading best model ...")
model.load_state_dict(torch.load("data/best_rnn_lm.pt"))

### END YOUR CODE

/Users/seebergerph/anaconda3/envs/seqlrn/lib/python3.10/site-packages/torch/nn/modules/rnn.py:83: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


Epoch 1 of 100
--------------------


Train: 100%|██████████| 2386/2386 [00:03<00:00, 651.63it/s]


Train loss: 8.2851 | Test loss: 7.2971 | Perplexity: 1476.0176
Epoch 2 of 100
--------------------


Train: 100%|██████████| 2386/2386 [00:03<00:00, 729.59it/s]


...

Train: 100%|██████████| 2386/2386 [00:03<00:00, 764.40it/s]


Train loss: 4.9992 | Test loss: 5.9308 | Perplexity: 376.4478
Epoch 100 of 100
--------------------


Train: 100%|██████████| 2386/2386 [00:03<00:00, 764.11it/s]


Train loss: 4.9924 | Test loss: 5.9316 | Perplexity: 376.7558
Best epoch: 67
Best test loss: 5.9166
Loading best model ...


<All keys matched successfully>

### Generate Titles

3.1 Use the trained RNN Language Model to generate theses titles. How can you sample the next tokens?

3.2 Compare your results with n-gram language models (e.g., n=4). Of course, you can use a library such as NLTK toolkit
- What perplexity does a regular 4-gram have on the same split? 
- Compare the generated titles from the 4-gram and RNN-LM. Do you think the n-gram titles are better?

In [11]:
### TODO: 3.1 Generate titles with the trained RNN Language Model

### YOUR CODE HERE

def generate(model, word2idx, idx2word, max_tokens, top_k, start_token="<s>", end_token="</s>"):
    model.eval()

    # Begin with start token and generate title
    # Notice: Ensure correct dimensions
    with torch.no_grad():
        hidden = None
        input = torch.tensor([word2idx[start_token]], dtype=torch.long)
        input = input.view(1,1)

        title = []
        for i in range(max_tokens):
            logits, hidden = model(input, hidden=hidden)
            logits = logits[0][0]

            # Prepare top-k predicted tokens
            topv, topi = torch.topk(logits, top_k)
            indices_to_remove = logits < topv[-1]
            logits[indices_to_remove] = -1 * float("Inf")

            # Sample from top-k predicted tokens
            pred_idx = torch.multinomial(F.softmax(logits, dim=0), 1)
            pred_idx = pred_idx.item()

            # Break if end of sentence token 
            if idx2word[pred_idx] == end_token:
                break

            # Prepare input for next forward pass
            input = torch.tensor([pred_idx], dtype=torch.long)
            input = input.view(1,1)

            # Append generated token to title
            title.append(idx2word[pred_idx])

        return title


for i in range(10):
    generated_title = generate(
        model=model,
        word2idx=word2idx,
        idx2word=idx2word,
        max_tokens=10,
        top_k=5
    )
    print(" ".join(generated_title))

### END YOUR CODE

entwurf und implementierung und implementierung und realisierung einer anwendung der
konzeption und bewertung von geschäftsprozessen für eine
konzeption einer anwendung zur unterstützung von methoden
konzeption einer webanwendung
analyse und prototypische implementierung
entwicklung eines konzepts für das
entwurf und prototypische implementierung von reifegradmodellen für das wissensmanagement und
entwicklung eines konzepts
konzeption und implementierung und prototypische umsetzung einer mobilen roboter
konzeption einer mobilen endgeräten


In [12]:
### TODO: 3.2 Generate titles with the trained n-gram language model

### YOUR CODE HERE

from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.tokenize import word_tokenize
from nltk.lm import Laplace

train_data, test_data = train_test_split(
    [word_tokenize(t.lower()) for t in dataframe["Titel"].values], 
    test_size=int(0.2 * len(dataframe)),
    shuffle=True, random_state=42
)

n = 4

ngram_train_data, ngram_vocab = padded_everygram_pipeline(n, train_data)
ngram_test_data, _ = padded_everygram_pipeline(n, test_data)

lm = Laplace(order=n)
lm.fit(ngram_train_data, ngram_vocab)

s = 0
for i, test in enumerate(ngram_test_data):
    p = lm.entropy(test)
    s += p

print ("Perplexity: {0:4f}\n".format(2**(s/(i+1))))

for i in range(10):
    generated_title = lm.generate(num_words=10, text_seed=["<s>"], random_seed=i)
    print(" ".join(generated_title))

### END YOUR CODE

Perplexity: 816.572507

konzeption und implementierung einer plattformunabhängigen grafischen entwicklungsumgebung für binokulare algorithmen
<s> null safety in modernen programmiersprachen </s> </s> </s> </s>
tensorfaktorisierung als ansatz für empfehlungssysteme </s> </s> </s> </s> </s>
<s> die erschließung zukunftsorientierter geschäftsfelder durch anwendung von e-business in
<s> <s> evaluierung der analytischen anwendung „ ibm cognos “
analyse und umsetzung eines webauftritts für einen fakultätsjahresbericht mittels wordpress
inhalts- und stimmungsanalyse von kundenfeedback im automotiveumfeld </s> </s> </s>
<s> <s> konzeption einer internetplattform zum crowdsourcing von haushaltsbesorgungen </s>
<s> verfahren zur visualisierung der arbeitsweise künstlicher neuronaler netze </s>
<s> <s> beschleunigung der visualisierung wasserbaulicher simulationen durch gitterreduktionsverfahren </s>
